# Trimer Case II:  QME

This page uses function defined in `trimer-case2-functions.ipynb`.  This notebook file must be in the same folder as this notebook.

In [1]:
# get the definition of functions.
%run trimer-case2-functions.ipynb

In [2]:
# parameter values
omega0 = 1
gamma0 = 1e-8
temperature = 0.86

k0,lambda0 = scale(omega0)

# distance between emitters 
a = 2*lambda0

NT = planck_dist(omega0,temperature)

v, label = pbasis()

**Operators**

In [3]:
sz, sp, sm = spin_ops()
H, Omega = hamiltonian(omega0,a)
Gamma, gamma, I0 = decay_rate(gamma0,a)
L, c_ops = collapse_ops(gamma,NT)

**Self Test 1 - Check if the emitters are too close**

We check if the distance between emitters is too short for standard QME method.  First, we check if the transition energies are too far from $\omega_{0}$.  If the relative shift is $>0.1$, the distance may be too small.


In [4]:
W, E, order = eigen_energies(H)

DE = transition_energies(E)

print("  channel        Delta E (relative shift)")
for k in range(3):
    x = DE[k]
    z = (x-omega0)/omega0
    print("Channel {0:1d} {1:6.3f}({2:6.3f}),  {3:6.3f}({4:6.3f}),  {5:6.3f}({6:6.3f})"
          .format(k,x[0],z[0],x[1],z[1],x[2],z[2]))

  channel        Delta E (relative shift)
Channel 0  0.571(-0.429),   1.000( 0.000),   1.429( 0.429)
Channel 1  1.215( 0.215),   1.000( 0.000),   0.785(-0.215)
Channel 2  1.215( 0.215),   1.000( 0.000),   0.785(-0.215)


We compare the steady state density operator with the Gibbs state.  They differ significantly, the emitter distance is too small.

In [5]:
# steady state by QME
rho = steadystate(H,c_ops)

Z = np.sum(np.exp(-E/temperature))

p = np.zeros(8)
q = np.zeros(8)
for k in range(8):
   p[k] = rho[k,k].real
   q[k] = np.exp(-E[k]/temperature)/Z

idx = np.argsort(p)

for k in range(8):
    print("|{0:3s}>  {1:8.4f}  {2:8.4f}".format(label[idx[k]], p[idx[k]], q[order[k]]))  

|eee>    0.0195    0.0190
|ege>    0.0529    0.0336
|eeg>    0.0529    0.0639
|gee>    0.0529    0.0639
|geg>    0.1437    0.0913
|egg>    0.1437    0.1737
|gge>    0.1437    0.1737
|ggg>    0.3907    0.3810


## Intensity

First, we plot the intensity from ($\theta=\pi/2,\phi=0$) to ($\theta=\pi/2,\phi=\pi$) through the north pole along the meridian circle.  The angle $\alpha$ in the plot is measured from the starting point.  The profile matches the theoretical prediction $\sin^2\theta$.

In [6]:
nmax= 100
phi=0
r = emitters_pos(a)
dtheta = np.pi/nmax
intensity = []
angle = []
for k in range(nmax+1):
    theta = np.pi/2 - k*dtheta
    if theta >= 0:
        phi = 0
    else:
        phi = np.pi
        
    nvect = detector_pos(theta,phi)
    q = k0*np.inner(nvect,r)
    q = np.exp(-1j*q)
    d_op = sum(q*sm)
    intensity.append(I0*np.sin(theta)**2*expect(d_op.dag()*d_op,rho))
    angle.append(np.pi/2-theta)

angle = np.array(angle)
y0=intensity[0]
theory = y0*np.sin(np.pi/2-angle)**2
angle = np.array(angle)/np.pi
plt.figure(figsize=(4,3))
plt.plot(angle,intensity,color='r',label="qme")
plt.plot(angle,theory,ls='--',color='g',label="theory")
plt.xlabel(r"$\alpha/\pi$")
plt.ylabel(r"Intensity")
plt.legend(loc=4)
plt.show()

NameError: name 'r' is not defined

Next, we place the detector in the plane of the triangle and change the azimuthal angle.   Theory predicts the intensity is nearly constant.  The plot agrees with it.

In [ ]:
r = emitters_pos(a)

# reference location
theta = np.pi/2

nmax= 360
dphi = 2*np.pi/nmax
intensity = []
angle = []
for k in range(nmax+1):
    phi = k*dphi
    nvect = detector_pos(theta,phi)
    q = np.array([k0*np.inner(nvect,r[0]),k0*np.inner(nvect,r[1]),k0*np.inner(nvect,r[2])])
    w = np.exp(-1j*q)
    d_op = w[0]*sm[0]+w[1]*sm[1]+w[2]*sm[2]
    intensity.append(I0*np.sin(theta)**2*expect(d_op.dag()*d_op,rho))
    angle.append(phi)

angle=np.array(angle)
plt.figure(figsize=(4,3))
plt.plot(angle/np.pi,intensity)
plt.xlabel(r"$\alpha/\pi$")
plt.ylabel(r"Intensity")
plt.show()

## First-order coherence function

We evaluate $g^{(1)}(\tau)$ at three different direction in the plane.  They are all identical due to the same reason as the intensity.

In [ ]:
tlist = np.linspace(0.0, 400.0*gamma0, 400)

def get_g1(theta,phi):
    nvect = detector_pos(theta,phi)
    q = k0*np.inner(nvect,r)
    q = np.exp(-1j*q)
    d_op = sum(q*sm)
    rho_c = rho*d_op.dag()
    result = mesolve(H,rho_c,tlist,c_ops,e_ops=[d_op])
    return np.real(np.asarray(result.expect[0]))/expect(d_op.dag()*d_op,rho)

theta=np.pi/2
phi=[0,np.pi/6,np.pi/3]
g1_1 = get_g1(theta, phi[0])
g1_2 = get_g1(theta, phi[1])
g1_3 = get_g1(theta, phi[2])
plt.figure(figsize=(4,3))
plt.plot(tlist,g1_1,color='r',label=rf"$\phi={phi[0]:.3f}$")
plt.plot(tlist,g1_2,color='b',label=rf"$\phi={phi[1]:.3f}$")
plt.plot(tlist,g1_3,color='g',label=rf"$\phi={phi[2]:.3f}$")
plt.xlabel(r"$\tau \gamma_{0}$")
plt.ylabel(r"$g^{(1)}(\tau)$")
plt.axhline(y=0,ls='--',color="k")
plt.legend(loc=4)
plt.text(20,0.80,rf"$\theta = {theta:.3f}$")
plt.show()

## Second-order coherence function

The no-delay coherence function is $\tfrac43$, which is independent of the detector location nor the size of the triangle.  The following results agree with the theory.  However, $g^{(2)}(\tau)$ for $\tau\ne 0$ depends both on the detector location and the size.

In [ ]:
tlist = np.linspace(0.0, 400.0*gamma0, 400)

def get_g2(theta,phi):
    nvect = detector_pos(theta,phi)
    q = k0*np.inner(nvect,r)
    q = np.exp(-1j*q)
    d_op = sum(q*sm)
    rho_c = d_op*rho*d_op.dag()
    result = mesolve(H,rho_c,tlist,c_ops,e_ops=[d_op.dag()*d_op])
    return np.real(np.asarray(result.expect[0]))/expect(d_op.dag()*d_op,rho)**2

theta=np.pi/2
phi=[0,np.pi/6,np.pi/3]
g2_1 = get_g2(theta, phi[0])
g2_2 = get_g2(theta, phi[1])
g2_3 = get_g2(theta, phi[2])
plt.figure(figsize=(4,3))
plt.plot(tlist,g2_1,color='r',label=rf"$\phi={phi[0]:.3f}$")
plt.plot(tlist,g2_2,color='b',label=rf"$\phi={phi[1]:.3f}$")
plt.plot(tlist,g2_3,color='g',ls='--',label=rf"$\phi={phi[2]:.3f}$")
plt.xlabel(r"$\tau \gamma_{0}$")
plt.ylabel(r"$g^{(2)}(\tau)$")
plt.ylim([0,1.6])
plt.axhline(y=1.0,ls='--',color="grey")
plt.axhline(y=4/3,ls='--',color="k")
plt.legend(loc=4)
plt.text(30,1.40,rf"$\theta = {theta:.3f}$")
plt.show()

In [ ]:
tlist = np.linspace(0.0, 400.0*gamma0, 400)

def get_g2(theta,phi):
    nvect = detector_pos(theta,phi)
    q = k0*np.inner(nvect,r)
    q = np.exp(-1j*q)
    d_op = sum(q*sm)
    rho_c = d_op*rho*d_op.dag()
    result = mesolve(H,rho_c,tlist,c_ops,e_ops=[d_op.dag()*d_op])
    return np.real(np.asarray(result.expect[0]))/expect(d_op.dag()*d_op,rho)**2

phi=0.0
theta=[0,np.pi/6,np.pi/3]
g2_1 = get_g2(theta[0], phi)
g2_2 = get_g2(theta[1], phi)
g2_3 = get_g2(theta[2], phi)
plt.figure(figsize=(4,3))
plt.plot(tlist,g2_1,color='r',label=rf"$\theta={theta[0]:.3f}$")
plt.plot(tlist,g2_2,color='b',label=rf"$\theta={theta[1]:.3f}$")
plt.plot(tlist,g2_3,color='g',label=rf"$\theta={theta[2]:.3f}$")
plt.xlabel(r"$\tau \gamma_{0}$")
plt.ylabel(r"$g^{(2)}(\tau)$")
plt.ylim([0,1.6])
plt.axhline(y=1.0,ls='--',color="grey")
plt.axhline(y=4/3,ls='--',color="k")
plt.legend(loc=4)
plt.text(30,1.40,rf"$\phi = {phi:.3f}$")
plt.show()